In [1]:
import os
import sys

import torch

PROJECT_ROOT = os.path.abspath(
    os.path.join(os.getcwd(), "..")
    if os.path.basename(os.getcwd()) == "notebooks"
    else os.getcwd()
)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from srcs.datasets.vicocktail import load_vicocktail
from srcs.nets.backend.ctc import ctc_decode
from srcs.nets.e2e import get_model
from srcs.spm.spm_train import ensure_unigram
from srcs.spm.text_transofm import TextTransform
from srcs.trainer.trainer import Trainer
from srcs.trainer.utils import (
    create_dataloader,
    load_config,
    move_batch,
    set_seed,
)

CONFIG_PATH = os.path.join(PROJECT_ROOT, "config.yaml")
CHECKPOINT_PATH = os.path.join(
    PROJECT_ROOT, "checkpoints", "finetune_vsr_12", "final"
)
TEST_FRACTION = 1.0
SAMPLE_COUNT = 3

config = load_config(CONFIG_PATH)
evaluation_config = config["evaluation"]
set_seed(config["training"]["seed"])

test_dataset = load_vicocktail(
    test_fraction=TEST_FRACTION,
    splits=("test",),
    seed=config["training"]["seed"],
)["test"]

model_path, units_path = ensure_unigram()
text_transform = TextTransform(model_path, units_path)
model = get_model(
    "auto-vsr",
    text_transform.vocab_size,
    checkpoint=CHECKPOINT_PATH,
)

amp = evaluation_config.get("amp", True)
test_dataloader = create_dataloader(
    test_dataset,
    text_transform,
    "test",
    evaluation_config,
)
trainer = Trainer(
    model=model,
    optimizer=None,
    scheduler=None,
    scaler=None,
    text_transform=text_transform,
    config=evaluation_config,
    amp=amp,
)

print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Device: {trainer.device}")
print(f"Test samples: {len(test_dataset)}")

d:\projects\VietnameseVSR\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Checkpoint: d:\projects\VietnameseVSR\checkpoints\finetune_vsr_12\final
Device: cuda
Test samples: 1167


In [2]:
def test_get_contexts_output(model, batch):
    model.eval()
    contexts = model.get_contexts(
        batch["videos"],
        batch["video_lengths"],
    )

    expected_names = {
        "visual_features",
        "h2_features",
        "h4_features",
        "logits",
        "input_lengths",
    }
    assert set(contexts) == expected_names

    batch_size, max_time = batch["videos"].shape[:2]
    encoder_dim = model.proj_encoder.out_features
    visual_dim = model.proj_encoder.in_features
    vocab_size = model.ctc.ctc_lo.out_features

    assert contexts["visual_features"].shape == (batch_size, max_time, visual_dim)
    assert contexts["h2_features"].shape == (batch_size, max_time, encoder_dim)
    assert contexts["h4_features"].shape == (batch_size, max_time, encoder_dim)
    assert contexts["logits"].shape == (batch_size, max_time, vocab_size)
    assert torch.equal(contexts["input_lengths"], batch["video_lengths"])
    assert all(not tensor.requires_grad for tensor in contexts.values())
    assert model._contexts is None

    return contexts


def test_get_contexts_reset(model, batch, previous_contexts):
    current_contexts = model.get_contexts(
        batch["videos"],
        batch["video_lengths"],
    )

    assert current_contexts["h2_features"] is not previous_contexts["h2_features"]
    assert current_contexts["h4_features"] is not previous_contexts["h4_features"]
    assert model._contexts is None

    return current_contexts


batch = move_batch(next(iter(test_dataloader)), trainer.device)
contexts = test_get_contexts_output(model, batch)
contexts = test_get_contexts_reset(model, batch, contexts)

for name, tensor in contexts.items():
    print(f"{name}: shape={tuple(tensor.shape)}, requires_grad={tensor.requires_grad}")

print("get_contexts tests passed.")

visual_features: shape=(8, 256, 512), requires_grad=False
h2_features: shape=(8, 256, 768), requires_grad=False
h4_features: shape=(8, 256, 768), requires_grad=False
logits: shape=(8, 256, 3002), requires_grad=False
input_lengths: shape=(8,), requires_grad=False
get_contexts tests passed.


In [3]:
metrics = trainer.run_one_epoch(
    test_dataloader,
    training=False,
    description="Testing",
)

print(f"Test loss: {metrics['loss']:.6f}")
print(f"Test WER: {metrics['wer']:.6f}")

Testing: 100%|██████████| 146/146 [00:57<00:00,  2.54it/s, loss=61.4844, wer=0.4838]

Test loss: 61.484355
Test WER: 0.483832


In [4]:
batch = move_batch(next(iter(test_dataloader)), trainer.device)
amp_enabled = amp and trainer.device.type == "cuda"
model.eval()

with torch.inference_mode():
    with torch.amp.autocast(
        device_type=trainer.device.type,
        dtype=torch.float16,
        enabled=amp_enabled,
    ):
        outputs = model(**batch)

token_ids = ctc_decode(
    outputs["logits"],
    outputs["input_lengths"],
    text_transform.blank_id,
)

for index in range(min(SAMPLE_COUNT, len(token_ids))):
    label_length = int(batch["label_lengths"][index])
    reference = text_transform.decode(
        batch["labels"][index, :label_length]
    )
    prediction = text_transform.decode(token_ids[index])

    print(f"Sample {index + 1}")
    print(f"Reference:  {reference}")
    print(f"Prediction: {prediction}")
    print()

Sample 1
Reference:  khi mà mình rõ được những cái điều mà mình ưu tiên trong cuộc sống thì mọi lựa chọn chúng mình đưa ra đều lấy nó làm kim chỉ nam để xác định là nên nói có hay là nó không học cách
Prediction: khi mà mình rõ được những cái điều mà mình mới ưu tiên cho cuộc sống cái mọi lựa con của mình đưa ra đều này đó làm gì nghĩ làm để xác định ra này hỏi rồi không học cách

Sample 2
Reference:  mười bài học mà mình học được trong năm nay mình đã chọn lọc ra và sẽ chia sẻ với bạn
Prediction: tốt được bài hồng mà mình không cộng nào này mình đã chọn không nào định trên thì sẻ với bài

Sample 3
Reference:  dơ sai ừn ợp seo lơn ninh khoa học về việc tự học trong cuốn sách này ngoài việc đề cập tới việc tự học thì nó cũng đề cập tới việc là
Prediction: trắng học seo với khoa học về việc tự học thông cũng sách là hóa về những gặp tới việc tự học thì cũng biết cảm thấy việc là

